# 09 — Benchmarking & Reporting
### CuSelect-AI · Campaign CU-001 · Final notebook in the original research roadmap

**Research question:** What is supported by the saved evidence and executed software checks, and what remains **unvalidated** about the proposed Cu–Ag CO₂RR research system?

**Epistemic contract:** An internally passing developer-authored test is **not** an independent benchmark; a geometrically sound slab is **not** an energy-minimized catalyst; a scientific claim extracted from a paper is **not** a model prediction; a prioritized task is **not** active learning with calibrated uncertainty.

**Scope:** This notebook audits and consolidates existing evidence across Notebooks 02–08. It does **not** rerun external literature searches, make API calls, train models, calculate energies, infer new FE values, create blinded test sets, or change approval statuses. It creates an in-memory report; writing the optional report to disk requires explicit manual opt-in.

**Inputs:** The project's previously exported, locally stored source files. These may be ignored by Git; preserve them separately. P005 is currently the only publication with approved observations. This notebook will **fail** if a required export is absent instead of silently inventing replacements.


## 9.1 — Reproducibility contract and exact source inventory

Read the authoritative human-review register, exported verified claims, source-paper register, and the manually exported geometry, descriptor, RAG, agent and critic artifacts. Keep the original strings intact. No writes occur in this section.


In [1]:
# PSEUDOCODE
# 1. Locate the CuSelect-AI repository using pyproject.toml.
# 2. Identify the existing authoritative and exported inputs from Notebooks 02–08.
# 3. Fail if any source is missing; do not manufacture missing records.
# 4. Load CSV/JSON/JSONL as strings or original JSON values.
# 5. Keep network, experiment, modeling, human approval, and export switches OFF.

from pathlib import Path
import hashlib
import json
import platform
import shutil
import tempfile
import pandas as pd
from IPython.display import display, Markdown

_here09 = Path.cwd().resolve()
ROOT = next((p for p in (_here09, *_here09.parents) if (p / 'pyproject.toml').is_file()), None)
assert ROOT is not None, 'Run inside the CuSelect-AI project (pyproject.toml required).'

paths09 = {
    'papers': ROOT / 'data/papers.csv',
    'human_reviews': ROOT / 'data/references/02_human_reviewed_claims.csv',
    'verified_claims': ROOT / 'data/processed/02_verified_claims.csv',
    'readiness02': ROOT / 'outputs/reports/02_evidence_readiness.json',
    'geometry_registry04': ROOT / 'data/processed/04_structures/04_structure_registry.csv',
    'geometry_manifest04': ROOT / 'data/processed/04_structures/04_structure_manifest.json',
    'features05': ROOT / 'data/processed/05_geometry_descriptors/05_geometry_descriptors.csv',
    'features_manifest05': ROOT / 'data/processed/05_geometry_descriptors/05_descriptor_manifest.json',
    'corpus06': ROOT / 'data/processed/06_rag_prototype/06_retrieval_corpus.jsonl',
    'corpus_manifest06': ROOT / 'data/processed/06_rag_prototype/06_retrieval_manifest.json',
    'agent_ledger07': ROOT / 'data/processed/07_multi_agent_prototype/07_agent_decision_ledger.jsonl',
    'agent_queue07': ROOT / 'data/processed/07_multi_agent_prototype/07_candidate_review_queue.csv',
    'agent_manifest07': ROOT / 'data/processed/07_multi_agent_prototype/07_agent_manifest.json',
    'critic_ledger08': ROOT / 'data/processed/08_scientific_critic_prototype/08_critic_decision_ledger.jsonl',
    'critic_tests08': ROOT / 'data/processed/08_scientific_critic_prototype/08_adversarial_development_tests.csv',
    'critic_queue08': ROOT / 'data/processed/08_scientific_critic_prototype/08_candidate_review_queue.csv',
    'research_triage08': ROOT / 'data/processed/08_scientific_critic_prototype/08_research_triage_queue.csv',
    'critic_manifest08': ROOT / 'data/processed/08_scientific_critic_prototype/08_critic_manifest.json',
}
missing09 = [str(p) for p in paths09.values() if not p.is_file()]
assert not missing09, 'Required prior notebook export is missing:\n' + '\n'.join(missing09)

csv_names09 = {
    'papers','human_reviews','verified_claims','geometry_registry04',
    'features05','agent_queue07','critic_tests08','critic_queue08','research_triage08'
}
json_names09 = {
    'readiness02','geometry_manifest04','features_manifest05',
    'corpus_manifest06','agent_manifest07','critic_manifest08'
}
frames09 = {k: pd.read_csv(paths09[k], dtype=str, keep_default_na=False) for k in csv_names09}
manifests09 = {k: json.loads(paths09[k].read_text(encoding='utf-8')) for k in json_names09}
def load_jsonl09(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
corpus09 = load_jsonl09(paths09['corpus06'])
agent_runs09 = load_jsonl09(paths09['agent_ledger07'])
critic_decisions09 = load_jsonl09(paths09['critic_ledger08'])

ALLOW_API09 = False
RUN_ENERGY09 = False
TRAIN_MODEL09 = False
EXECUTE_EXPERIMENT09 = False
UPDATE_HUMAN_APPROVAL09 = False
EXPORT_REPORT09 = False
assert all(not x for x in (ALLOW_API09,RUN_ENERGY09,TRAIN_MODEL09,EXECUTE_EXPERIMENT09,UPDATE_HUMAN_APPROVAL09,EXPORT_REPORT09))
assert manifests09['readiness02']['campaign_id'] == 'CU-001'
print('NOTEBOOK 09 — REPRODUCIBILITY CONTRACT: PASSED')
print('Project:', ROOT)
print('Python:', platform.python_version(), '| required source files:', len(paths09))
print('Network / computation / experiment / approvals / report export: DISABLED')
print('No files modified')


NOTEBOOK 09 — REPRODUCIBILITY CONTRACT: PASSED
Project: /Users/nsolebna/202609_Ironhack/Week 8/Project 3/CuSelect-AI
Python: 3.11.18 | required source files: 18
Network / computation / experiment / approvals / report export: DISABLED
No files modified


## 9.2 — Cryptographic chain of custody and human-reviewed evidence synchronization

The point of this test is **identity preservation**, not scientific truth. Recompute saved SHA-256 digests and compare all 17 fields of the approved experimental records. Verify that the RAG corpus, agent ledger and critic ledger are the same bytes attested by their manifests. Confirm that the critic references the saved agent ledger.


In [2]:
# PSEUDOCODE
# 1. Recompute and verify the original RAG, agent, and critic ledger file digests.
# 2. Verify that agent/critic manifests refer to the same upstream corpus/ledger hashes.
# 3. Compare full records in the human-reviewed register and verified export.
# 4. Confirm corpus evidence IDs, DOIs, and source locations match the approved source.
# 5. Preserve benchmark readiness as false; do not modify any files.

def sha_file09(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha_file09(paths09['corpus06']) == manifests09['corpus_manifest06']['corpus_sha256']
assert sha_file09(paths09['agent_ledger07']) == manifests09['agent_manifest07']['agent_ledger_sha256']
assert sha_file09(paths09['critic_ledger08']) == manifests09['critic_manifest08']['critic_ledger_sha256']
assert manifests09['agent_manifest07']['source_corpus_sha256'] == manifests09['corpus_manifest06']['corpus_sha256']
assert manifests09['critic_manifest08']['agent_ledger_sha256'] == manifests09['agent_manifest07']['agent_ledger_sha256']

papers09 = frames09['papers']; reviews09 = frames09['human_reviews']; claims09 = frames09['verified_claims']
assert papers09['paper_id'].is_unique and reviews09['evidence_id'].is_unique and claims09['evidence_id'].is_unique
approved09 = reviews09.loc[
    reviews09['review_status'].eq('human_verified') &
    reviews09['source_checked_by_human'].str.strip().str.lower().eq('true')
].copy()
assert set(approved09.columns) == set(claims09.columns)
all_fields09 = sorted(claims09.columns)
pd.testing.assert_frame_equal(
    approved09[all_fields09].sort_values('evidence_id').reset_index(drop=True),
    claims09[all_fields09].sort_values('evidence_id').reset_index(drop=True),
    check_dtype=False,
)
assert len(approved09) == len(claims09) == len(corpus09) == 3
assert len(all_fields09) == 17
by_claim09 = {r['evidence_id']:r for r in claims09.to_dict('records')}
assert len({r['evidence_id'] for r in corpus09}) == 3
paper_doi09 = dict(zip(papers09['paper_id'],papers09['doi'].str.strip().str.lower()))
for r in corpus09:
    src = by_claim09[r['evidence_id']]
    assert r['paper_id'] == src['paper_id'] and r['doi'].lower().strip() == src['doi'].lower().strip()
    assert r['source_location'] == src['exact_source_location']
    assert paper_doi09[r['paper_id']] == r['doi'].lower().strip()
    for field in ('claim_paraphrase','catalyst_system','conditions','source_extent','fe_product','faradaic_efficiency_percent'):
        assert str(r[field]) == str(src[field]), (r['evidence_id'],field)
    assert r['review_status'] == 'human_verified'
assert manifests09['readiness02']['evidence_ready_for_rag'] is False
assert manifests09['readiness02']['ready_for_blinded_benchmark'] is False
assert manifests09['corpus_manifest06']['production_RAG_validated'] is False
assert manifests09['critic_manifest08']['independent_critic_benchmark_completed'] is False
print('CROSS-NOTEBOOK CHAIN OF CUSTODY: PASSED')
print('Human-approved claims:',len(approved09),'| synchronized fields:',len(all_fields09))
print('RAG, agent, critic manifest SHA-256: MATCHED')
print('Scientific benchmark readiness: NOT ESTABLISHED')


CROSS-NOTEBOOK CHAIN OF CUSTODY: PASSED
Human-approved claims: 3 | synchronized fields: 17
RAG, agent, critic manifest SHA-256: MATCHED
Scientific benchmark readiness: NOT ESTABLISHED


## 9.3 — Evidence breadth and measurement-type coverage

Count measurements by their **actual meaning**. The ratio observation is not an FE row. A percentage of eight preselected seed publications must not be mistaken for coverage of the global research literature. With only one independent evidence-bearing publication, a DOI-held-out test is **not estimable**.


In [4]:

# PSEUDOCODE
# 1. Verify that the publication register contains eight seed papers.
# 2. Determine how many independent publications contain approved claims.
# 3. Distinguish structured Faradaic efficiencies from product-ratio data.
# 4. Calculate the current scientific evidence coverage.
# 5. Map these counts to the actual Notebook 02 readiness-report keys.
# 6. Compare independently calculated counts with saved report values.
# 7. Confirm that DOI-held-out benchmarking is not yet possible.
# 8. Display the results without modifying any files.

import pandas as pd
from IPython.display import display

# --------------------------------------------------
# STEP 1 — Validate publication coverage
# --------------------------------------------------

assert len(papers09) == 8, (
    "Expected eight seed publications."
)

paper_ids09 = set(claims09["paper_id"])

assert len(paper_ids09) == 1, (
    "Unexpected number of independent publications "
    "with approved scientific evidence."
)

# --------------------------------------------------
# STEP 2 — Identify different measurement types
# --------------------------------------------------

fe_values09 = (
    claims09["faradaic_efficiency_percent"]
    .astype(str)
    .str.strip()
)

n_structured_fe09 = int(
    fe_values09.ne("").sum()
)

n_non_fe09 = int(
    fe_values09.eq("").sum()
)

assert n_structured_fe09 == 2, (
    "Expected two structured FE observations."
)

assert n_non_fe09 == 1, (
    "Expected one non-FE observation."
)

# The product-ratio observation must not be
# incorrectly represented as a Faradaic efficiency.

assert (
    claims09.loc[
        fe_values09.eq(""),
        "fe_product"
    ]
    .astype(str)
    .str.strip()
    .eq("")
    .all()
), "A non-FE observation contains an unexpected FE product."

# --------------------------------------------------
# STEP 3 — Compute evidence coverage
# --------------------------------------------------

coverage09 = {
    "seed_publications": len(papers09),
    "publications_with_approved_claims": len(paper_ids09),
    "publications_without_approved_claims": (
        len(papers09) - len(paper_ids09)
    ),
    "approved_claims": len(claims09),
    "structured_FE_observations": n_structured_fe09,
    "other_observations_not_structured_as_FE": n_non_fe09,
    "retrieval_records": len(corpus09),
    "doi_held_out_test_possible": False,
}

# --------------------------------------------------
# STEP 4 — Map coverage to Notebook 02 report keys
# --------------------------------------------------

# IMPORTANT:
# Notebook 02 uses "seed_papers", not "seed_publications".
# We must preserve the original report schema.

readiness_checks09 = {
    "seed_papers": coverage09["seed_publications"],
    "publications_with_verified_claims": (
        coverage09["publications_with_approved_claims"]
    ),
    "publications_without_verified_claims": (
        coverage09["publications_without_approved_claims"]
    ),
    "human_verified_claims": coverage09["approved_claims"],
    "verified_claims_eligible_for_retrieval": (
        coverage09["retrieval_records"]
    ),
}

# --------------------------------------------------
# STEP 5 — Validate against the saved readiness report
# --------------------------------------------------

readiness_report09 = manifests09["readiness02"]

for report_key, expected in readiness_checks09.items():

    assert report_key in readiness_report09, (
        f"Missing readiness-report field: {report_key}"
    )

    actual = readiness_report09[report_key]

    assert actual == expected, (
        f"Readiness mismatch for {report_key}: "
        f"expected {expected}, found {actual}"
    )

# --------------------------------------------------
# STEP 6 — Preserve scientific benchmark restrictions
# --------------------------------------------------

assert not coverage09["doi_held_out_test_possible"], (
    "A DOI-held-out benchmark cannot be established "
    "from one approved evidence-bearing publication."
)

# --------------------------------------------------
# STEP 7 — Display validated results
# --------------------------------------------------

print("NOTEBOOK 09 — EVIDENCE COVERAGE AUDIT")
print("=" * 52)

coverage_table09 = pd.DataFrame(
    [
        {"metric": key, "value": value}
        for key, value in coverage09.items()
    ]
)

display(coverage_table09)

print("\nReadiness-report reconciliation: PASSED")
print("Structured FE observations:", n_structured_fe09)
print("Non-FE observations:", n_non_fe09)

print(
    "\nEVIDENCE COVERAGE AUDIT: PASSED | "
    "DOI-held-out benchmark: NOT ESTIMABLE"
)

print("No source files or readiness reports modified.")


NOTEBOOK 09 — EVIDENCE COVERAGE AUDIT


,metric,value
0,seed_publications,8
1,publications_with_approved_claims,1
2,publications_without_approved_claims,7
3,approved_claims,3
4,structured_FE_observations,2
5,other_observations_not_structured_as_FE,1
6,retrieval_records,3
7,doi_held_out_test_possible,False



Readiness-report reconciliation: PASSED
Structured FE observations: 2
Non-FE observations: 1

EVIDENCE COVERAGE AUDIT: PASSED | DOI-held-out benchmark: NOT ESTIMABLE
No source files or readiness reports modified.


## 9.4 — Geometry-feature lineage and computational readiness

Eight idealized Cu/Ag slabs are geometry *inputs*, not independent reference labels or predicted catalyst rankings. Check alignment of the Notebook 04 registry and Notebook 05 descriptors by candidate ID and full geometry fingerprint; preserve pending human-visual-review and no-modeling-approval statuses.


In [5]:
# PSEUDOCODE
# 1. Reconcile the eight structure IDs and their full geometry fingerprints.
# 2. Verify unrelaxed and pending-review status for every structure.
# 3. Check that descriptors contain no FE, DFT, or EMT/MACE energy labels.
# 4. Verify registered EXTXYZ files exist; do not perform geometry relaxation.
# 5. Report computational readiness conservatively.

registry09 = frames09['geometry_registry04']
features09 = frames09['features05']
assert len(registry09) == len(features09) == 8
assert registry09['candidate_id'].is_unique and features09['candidate_id'].is_unique
assert set(registry09['candidate_id']) == set(features09['candidate_id'])
assert registry09['geometry_sha256'].is_unique
reg_hash09 = dict(zip(registry09['candidate_id'],registry09['geometry_sha256']))
assert all(r['geometry_sha256'] == reg_hash09[r['candidate_id']] for r in features09.to_dict('records'))
assert all(manifests09['features_manifest05']['fingerprints'][cid] == val for cid,val in reg_hash09.items())
assert registry09['geometry_status'].eq('unrelaxed').all()
assert registry09['structure_review_status'].eq('pending_visual_review').all()
assert not registry09['approved_for_modeling'].str.lower().eq('true').any()
assert not features09['approved_for_modeling'].str.lower().eq('true').any()
assert features09['descriptor_scope'].eq('geometry_only_unrelaxed').all()
assert manifests09['geometry_manifest04']['ml_approved'] is False
assert manifests09['features_manifest05']['modeling_approved'] is False
for forbidden in ('DFT_reference_labels_included','experimental_selectivity_targets_included','EMT_or_MACE_energies_included'):
    assert manifests09['features_manifest05'][forbidden] is False
for cid in reg_hash09:
    assert (paths09['geometry_registry04'].parent / f'{cid}.extxyz').is_file(), cid
geometries09 = {
    'registered_models':len(registry09), 'descriptor_rows':len(features09),
    'reference_DFT_for_this_geometry_set':False,
    'human_visual_review_completed':False,
    'approved_for_predictive_modeling':False,
    'catalytic_selectivity_predictions_available':False,
}
display(pd.DataFrame([{'metric':k,'value':v} for k,v in geometries09.items()]))
print('GEOMETRY–FEATURE PROVENANCE: PASSED | Computational validity: PENDING')


,metric,value
0,registered_models,8
1,descriptor_rows,8
2,reference_DFT_for_this_geometry_set,False
3,human_visual_review_completed,False
4,approved_for_predictive_modeling,False
5,catalytic_selectivity_predictions_available,False


GEOMETRY–FEATURE PROVENANCE: PASSED | Computational validity: PENDING


## 9.5 — Multi-agent coordination and execution trace audit

Replay the *saved* three Notebook 07 agent runs, not new agent calls. Check that the three bounded steps refer to the intended role and tool, and that every cited claim and proposed candidate is registered. This evaluates recorded developer examples only.


In [6]:
# PSEUDOCODE
# 1. Verify each agent-run hash from its canonical JSON payload.
# 2. Confirm a fixed three-step Literature → Design → Critic trace.
# 3. Check the three recorded decisions and citations against admitted records.
# 4. Verify no external API, energy calculation, selectivity prediction or approval.
# 5. Display the recorded decisions without executing the agents.

def canonical09(value):
    return json.dumps(value,sort_keys=True,separators=(',',':'),ensure_ascii=False,default=str)
def digest09(value):
    return hashlib.sha256(canonical09(value).encode('utf-8')).hexdigest()
assert len(agent_runs09) == manifests09['agent_manifest07']['run_count'] == 3
expected_roles09 = [
    ('literature_agent','approved_claim_lookup'),
    ('design_agent','registered_geometry_lookup'),
    ('scientific_critic','policy_check'),
]
valid_claims09 = set(by_claim09)
valid_geometries09 = set(reg_hash09)
agent_rows09 = []
for run in agent_runs09:
    assert run['run_sha256'] == digest09({k:v for k,v in run.items() if k != 'run_sha256'})
    assert run['source_corpus_sha256'] == manifests09['corpus_manifest06']['corpus_sha256']
    assert len(run['trace']) == manifests09['agent_manifest07']['tool_steps_per_run'] == 3
    assert [(e['agent'],e['tool']) for e in run['trace']] == expected_roles09
    assert [e['step'] for e in run['trace']] == [1,2,3]
    assert set(run['critic']['citation_ids']) <= valid_claims09
    assert set(run['critic']['candidate_ids']) <= valid_geometries09
    assert run['critic']['modeling_approved'] is False
    assert run['predicted_selectivity'] is False
    assert run['human_approval_modified'] is False
    assert run['external_api_called'] is False
    agent_rows09.append({'task_id':run['task']['task_id'],
                         'request_type':run['task']['task_type'],
                         'decision':run['critic']['decision'],
                         'tool_steps':len(run['trace'])})
assert [r['decision'] for r in agent_rows09] == ['evidence_only','hypothesis_only','abstain']
assert manifests09['agent_manifest07']['human_approvals_modified'] is False
assert manifests09['agent_manifest07']['predicted_catalytic_selectivity'] is False
agent_audit09 = pd.DataFrame(agent_rows09)
display(agent_audit09)
print('SAVED AGENT WORKFLOW AUDIT: PASSED (3 developer-authored demonstrations)')
print('The 10 Notebook 07 development tests are not independently rerun here.')


,task_id,request_type,decision,tool_steps
0,W07-01,source_question,evidence_only,3
1,W07-02,structure_hypothesis,hypothesis_only,3
2,W07-03,predict_selectivity,abstain,3


SAVED AGENT WORKFLOW AUDIT: PASSED (3 developer-authored demonstrations)
The 10 Notebook 07 development tests are not independently rerun here.


## 9.6 — Critic decision accounting: positive vs adversarial development cases

The **four positive** and **twenty negative** cases were authored during development. Their saved decisions are auditable, but **not an unbiased held-out benchmark**. Compute observed false-advancement and false-rejection counts *only for these cases*. Do not generalize the resulting rates to scientific deployment or future LLM outputs.


In [7]:
# PSEUDOCODE
# 1. Recompute the integrity hash of the exported critic decision ledger.
# 2. Match all saved adversarial decisions to the exported development-case table.
# 3. Verify the four positive cases were admitted narrowly, never modeling-approved.
# 4. Calculate false advancements and false rejections on the authored cases.
# 5. Label resulting rates as development-test results, not independent reliability estimates.

attack_tests09 = frames09['critic_tests08']
assert len(critic_decisions09) == manifests09['critic_manifest08']['critic_decisions'] == 24
assert len(attack_tests09) == manifests09['critic_manifest08']['adversarial_dev_cases'] == 20
assert attack_tests09['case'].is_unique
by_decision09 = {x['assertion_id']:x for x in critic_decisions09}
assert len(by_decision09) == len(critic_decisions09)
positive_ids09 = {f'P08-{i:02d}' for i in range(1,5)}
negative_ids09 = {f'A08-{i:02d}' for i in range(1,21)}
assert set(by_decision09) == positive_ids09 | negative_ids09
expected_positive09 = {'P08-01':'evidence_only','P08-02':'evidence_only',
                       'P08-03':'evidence_only','P08-04':'hypothesis_only'}
false_reject09 = sum(by_decision09[i]['decision'] != d for i,d in expected_positive09.items())
false_advance09 = sum(by_decision09[i]['decision'] != 'abstain' for i in negative_ids09)
for x in critic_decisions09:
    assert x['scientific_modeling_approved'] is False
    assert set(x['cited_ids']) <= valid_claims09
    assert set(x['candidate_ids']) <= valid_geometries09
for row in attack_tests09.to_dict('records'):
    case_id = row['case'].split('_',1)[0]
    assert by_decision09[case_id]['decision'] == row['decision']
    assert row['correctly_abstained'].strip().lower() == str(row['decision']=='abstain').lower()
assert false_advance09 == manifests09['critic_manifest08']['false_advancements_in_authored_cases'] == 0
assert false_reject09 == 0
critic_dev_metrics09 = {
    'positive_authored_cases':len(positive_ids09),
    'positive_authored_cases_passed':len(positive_ids09)-false_reject09,
    'adversarial_authored_cases':len(negative_ids09),
    'adversarial_authored_cases_passed':len(negative_ids09)-false_advance09,
    'false_rejections_authored_positive':false_reject09,
    'false_advancements_authored_negative':false_advance09,
    'independent_scientific_benchmark':False,
}
display(pd.DataFrame([{'metric':k,'value':v} for k,v in critic_dev_metrics09.items()]))
print('CRITIC DEVELOPMENT CASE ACCOUNTING: PASSED (4 positive, 20 adversarial)')
print('These are NOT external/generalization performance estimates.')


,metric,value
0,positive_authored_cases,4
1,positive_authored_cases_passed,4
2,adversarial_authored_cases,20
3,adversarial_authored_cases_passed,20
4,false_rejections_authored_positive,0
5,false_advancements_authored_negative,0
6,independent_scientific_benchmark,False


CRITIC DEVELOPMENT CASE ACCOUNTING: PASSED (4 positive, 20 adversarial)
These are NOT external/generalization performance estimates.


## 9.7 — Honest evaluation scorecard across scientific claims and engineering functions

Different categories require different standards. The saved RAG corpus and agent traces establish a working offline baseline. They do **not** include an independently labeled retrieval benchmark, semantic-embedding comparison, calibrated uncertainty, experimental comparison, or reference DFT benchmark. Notebook 06's **6/6 smoke tests** and Notebook 07's **10/10 orchestration tests** were observed in those notebooks but are **not reverified from separate exported case-level ledgers here**; they must never be converted into a production quality score.


In [8]:
# PSEUDOCODE
# 1. Build an evidence-based scorecard of what has and has not been evaluated.
# 2. Keep recomputed developer-test results separate from earlier notebook self-reports.
# 3. Use 'not_established' rather than inventing zeros for unavailable scientific scores.
# 4. Show the level of evidence for each metric.

scorecard09 = pd.DataFrame([
    {'dimension':'Experimental source provenance','observed':'3 exact full-record matches','basis':'Recomputed now','scientific_readiness':'limited to 1 source paper'},
    {'dimension':'Atomic structure lineage','observed':'8/8 fingerprints aligned','basis':'Recomputed now','scientific_readiness':'unrelaxed; human review pending'},
    {'dimension':'RAG lexical lookup','observed':'3 source-linked records','basis':'Corpus export + Notebook 06 prototype','scientific_readiness':'independent retrieval eval absent'},
    {'dimension':'RAG smoke-test cases','observed':'6/6 reported in Notebook 06','basis':'Prior developer-run output; not separately replayed','scientific_readiness':'not a held-out benchmark'},
    {'dimension':'Agent orchestration','observed':'3/3 trace hashes and roles verified','basis':'Recomputed from saved runs','scientific_readiness':'deterministic demo only'},
    {'dimension':'Agent smoke-test cases','observed':'10/10 reported in Notebook 07','basis':'Prior developer-run output; not separately replayed','scientific_readiness':'not an independent benchmark'},
    {'dimension':'Critic development cases','observed':f'{4-false_reject09}/4 positive; {20-false_advance09}/20 adversarial',
     'basis':'Recomputed from saved developer-authored decisions','scientific_readiness':'out-of-sample validity unknown'},
    {'dimension':'Energy/force prediction','observed':'not performed','basis':'Source manif./restriction checks','scientific_readiness':'reference DFT and MLIP validation absent'},
    {'dimension':'CO2RR FE/selectivity prediction','observed':'not performed','basis':'Science and approval gates','scientific_readiness':'no mapping or benchmark'},
    {'dimension':'Bayesian active learning','observed':'not implemented','basis':'Notebook 08 triage manifest','scientific_readiness':'heuristic worklist only'},
])
display(scorecard09)
assert not any('validated' in str(r['observed']).lower() for r in scorecard09.to_dict('records'))
print('EVALUATION SCORECARD: GENERATED — scientific claims not inflated')


,dimension,observed,basis,scientific_readiness
0,Experimental source provenance,3 exact full-record matches,Recomputed now,limited to 1 source paper
1,Atomic structure lineage,8/8 fingerprints aligned,Recomputed now,unrelaxed; human review pending
2,RAG lexical lookup,3 source-linked records,Corpus export + Notebook 06 prototype,independent retrieval eval absent
3,RAG smoke-test cases,6/6 reported in Notebook 06,Prior developer-run output; not separately rep...,not a held-out benchmark
4,Agent orchestration,3/3 trace hashes and roles verified,Recomputed from saved runs,deterministic demo only
5,Agent smoke-test cases,10/10 reported in Notebook 07,Prior developer-run output; not separately rep...,not an independent benchmark
6,Critic development cases,4/4 positive; 20/20 adversarial,Recomputed from saved developer-authored decis...,out-of-sample validity unknown
7,Energy/force prediction,not performed,Source manif./restriction checks,reference DFT and MLIP validation absent
8,CO2RR FE/selectivity prediction,not performed,Science and approval gates,no mapping or benchmark
9,Bayesian active learning,not implemented,Notebook 08 triage manifest,heuristic worklist only


EVALUATION SCORECARD: GENERATED — scientific claims not inflated


## 9.8 — Independent benchmark eligibility and pre-registration checklist

**Benchmark validity requires:** multiple *independent, reviewed* publications; held-out evaluation queries and decisions frozen before implementation changes; citation-support labels judged by an independent domain reviewer; leak-safe DOI/structure-family grouping; and predefined measures of false advancement, false rejection, abstention, citation accuracy, coverage, and computational cost.

No train/test split is generated here. The benchmark is **not estimable with a one-paper corpus**. A benchmark plan is not a completed benchmark.


In [9]:
# PSEUDOCODE
# 1. Create a preregistration checklist for future independently labeled evaluation.
# 2. Fail closed when fewer than two independent evidence-bearing publications exist.
# 3. Represent unmeasured metrics explicitly as None / not_established.
# 4. Do not generate pseudo-test examples or split correlated records across train/test.

benchmark_plan09 = {
    'status':'blocked_pending_independent_evidence_and_pre_registration',
    'independent_reviewed_publications_available':len(paper_ids09),
    'doi_held_out_evaluation_possible':len(paper_ids09) >= 2,
    'held_out_split_created':False,
    'requirements':[
        'multiple independently reviewed source publications',
        'pre-registered domain-specific test questions and answer labels',
        'independent annotation of evidence support and critic decisions',
        'no overlap of DOI or structure families between development and test',
        'frozen model, retrieval index, prompt, policy, and cutoff before scoring',
        'record token/cost/latency together with abstention and error types',
    ],
    'independent_retrieval_precision_at_k':None,
    'independent_citation_support_rate':None,
    'independent_false_advancement_rate':None,
    'independent_false_rejection_rate':None,
    'independent_abstention_accuracy':None,
    'independent_selectivity_prediction_error':None,
}
assert benchmark_plan09['doi_held_out_evaluation_possible'] is False
assert benchmark_plan09['held_out_split_created'] is False
assert all(v is None for k,v in benchmark_plan09.items() if k.startswith('independent_') and k not in ('independent_reviewed_publications_available',))
display(pd.DataFrame([{'requirement':r,'completed':False} for r in benchmark_plan09['requirements']]))
print('INDEPENDENT BENCHMARK ELIGIBILITY: BLOCKED — 1 independent evidence-bearing publication')


,requirement,completed
0,multiple independently reviewed source publica...,False
1,pre-registered domain-specific test questions ...,False
2,independent annotation of evidence support and...,False
3,no overlap of DOI or structure families betwee...,False
4,"frozen model, retrieval index, prompt, policy,...",False
5,record token/cost/latency together with absten...,False


INDEPENDENT BENCHMARK ELIGIBILITY: BLOCKED — 1 independent evidence-bearing publication


## 9.9 — Error taxonomy, scientific boundaries and research-risk register

Classify the remaining gaps as explicit research requirements, not software failures. These risks should carry forward into any public demonstration or commercialization discussion.


In [10]:
# PSEUDOCODE
# 1. Define auditable research blockers and their consequences.
# 2. Tie each blocker to specific observed evidence or missing validation.
# 3. State the corrective scientific action without claiming completion.
# 4. Produce a stable in-memory risk register.

risk_rows09 = [
    {'risk_id':'RK09-01','domain':'Literature coverage','evidence':'3 approved claims from only P005',
     'risk':'No DOI-held-out generalization evidence','required_action':'Independently review additional publications','status':'open'},
    {'risk_id':'RK09-02','domain':'Experimental comparability','evidence':'Different products/potentials; 1 ratio observation',
     'risk':'Invalid cross-catalyst selectivity ranking','required_action':'Human-verify structured operating conditions and comparable targets','status':'open'},
    {'risk_id':'RK09-03','domain':'Geometry applicability','evidence':'8 idealized, unrelaxed slab models',
     'risk':'Surface stability and working-state geometry unknown','required_action':'Human inspect layers, periodic motifs and slab construction','status':'open'},
    {'risk_id':'RK09-04','domain':'Computational validity','evidence':'No independent DFT reference energies/forces for candidate set',
     'risk':'Energy and force predictions not benchmarked','required_action':'Reference DFT with convergence and model applicability analysis','status':'open'},
    {'risk_id':'RK09-05','domain':'RAG scientific support','evidence':'Lexical retrieval over three verified records',
     'risk':'Poor unseen-query coverage and citation support unquantified','required_action':'Source-reviewed expansion and frozen, independently labeled retrieval evaluation','status':'open'},
    {'risk_id':'RK09-06','domain':'Critic generalization','evidence':'4 authored positive + 20 authored adversarial cases',
     'risk':'Overfitting to development checks','required_action':'Blind independent scientific adversarial evaluation','status':'open'},
    {'risk_id':'RK09-07','domain':'Agent autonomy','evidence':'Offline deterministic roles; no generative LLM',
     'risk':'No demonstrated autonomous, robust scientific reasoning','required_action':'Isolated model-backed tests with strict citations and human oversight','status':'open'},
    {'risk_id':'RK09-08','domain':'Active-learning claims','evidence':'6 heuristic research actions; zero executed',
     'risk':'Triage scores may be mistaken for information gain','required_action':'Build validated surrogate, uncertainty calibration and acquisition evaluation','status':'open'},
]
risk_register09 = pd.DataFrame(risk_rows09)
assert risk_register09['risk_id'].is_unique and risk_register09['status'].eq('open').all()
display(risk_register09[['risk_id','domain','risk','required_action','status']])
print('SCIENTIFIC RISK REGISTER:',len(risk_register09),'open research gaps; no automatic approvals')


,risk_id,domain,risk,required_action,status
0,RK09-01,Literature coverage,No DOI-held-out generalization evidence,Independently review additional publications,open
1,RK09-02,Experimental comparability,Invalid cross-catalyst selectivity ranking,Human-verify structured operating conditions a...,open
2,RK09-03,Geometry applicability,Surface stability and working-state geometry u...,"Human inspect layers, periodic motifs and slab...",open
3,RK09-04,Computational validity,Energy and force predictions not benchmarked,Reference DFT with convergence and model appli...,open
4,RK09-05,RAG scientific support,Poor unseen-query coverage and citation suppor...,"Source-reviewed expansion and frozen, independ...",open
5,RK09-06,Critic generalization,Overfitting to development checks,Blind independent scientific adversarial evalu...,open
6,RK09-07,Agent autonomy,"No demonstrated autonomous, robust scientific ...",Isolated model-backed tests with strict citati...,open
7,RK09-08,Active-learning claims,Triage scores may be mistaken for information ...,"Build validated surrogate, uncertainty calibra...",open


SCIENTIFIC RISK REGISTER: 8 open research gaps; no automatic approvals


## 9.10 — Research-action audit and next-step handoff

The Notebook 08 queue contains six **developer-authored** actions with ordinal *planning* scores, not posterior uncertainty or expected improvement. Preserve their blocked/prerequisite statuses and do not execute tasks automatically.


In [11]:
# PSEUDOCODE
# 1. Read and validate the six research-prioritization tasks from Notebook 08.
# 2. Verify they are planning suggestions, not completed calculations or experiments.
# 3. Preserve existing ordering and prerequisite restrictions.
# 4. Display the next actionable research tasks without generating new priorities.

triage09 = frames09['research_triage08'].copy()
assert len(triage09) == manifests09['critic_manifest08']['research_actions'] == 6
assert triage09['action_id'].is_unique
assert set(triage09['status']) == {'propose_for_manual_planning','blocked_pending_prerequisites'}
for kind in ('reference_calculation_request','benchmark_design'):
    assert triage09.loc[triage09['kind'].eq(kind),'status'].eq('blocked_pending_prerequisites').all()
assert manifests09['critic_manifest08']['research_actions_executed'] == 0
assert manifests09['critic_manifest08']['active_learning_acquisition_implemented'] is False
assert manifests09['critic_manifest08']['human_approval_modified'] is False
assert all(triage09['prerequisites'].astype(str).str.len() > 10)
display(triage09[['action_id','action','kind','triage_score_not_scientific','status']])
print('RESEARCH HANDOFF: six suggested tasks | zero executed | no uncertainty scores')


,action_id,action,kind,triage_score_not_scientific,status
0,R08-01,Human-review independent Cu–Ag CO2RR experimen...,source_review,9,propose_for_manual_planning
1,R08-02,Complete human visual review of eight exported...,geometry_review,9,propose_for_manual_planning
2,R08-03,Structure experimental condition metadata for ...,condition_curation,8,propose_for_manual_planning
3,R08-06,Review pretrained MLIP Cu/Ag surface domain ap...,model_applicability_review,4,blocked_pending_prerequisites
4,R08-04,Acquire independent reference DFT energies/for...,reference_calculation_request,3,blocked_pending_prerequisites
5,R08-05,Design DOI-held-out retrieval and critic evalu...,benchmark_design,3,blocked_pending_prerequisites


RESEARCH HANDOFF: six suggested tasks | zero executed | no uncertainty scores


## 9.11 — Construct an evidence-backed report and machine-readable summary **in memory**

The report below distinguishes observed software checks, authored development cases, and absent scientific benchmarks. It is a factual *status report*, not an endorsement of catalyst performance. Dates, author names and subjective market claims are intentionally not invented.


In [12]:
# PSEUDOCODE
# 1. Create a machine-readable factual summary from recomputed data and saved manifests.
# 2. Clearly mark scientifically absent benchmarks and prediction targets.
# 3. Assemble a concise Markdown report with a development-versus-validation distinction.
# 4. Keep both documents in memory until the explicit export gate is enabled.

summary09 = {
    'campaign_id':'CU-001',
    'notebook':'09_Benchmarking_and_Reporting',
    'benchmark_status':'initial_cross_notebook_audit_only',
    'source_publications_seeded':coverage09['seed_publications'],
    'independent_publications_with_verified_claims':coverage09['publications_with_approved_claims'],
    'human_verified_experimental_claims':coverage09['approved_claims'],
    'structured_FE_claims':coverage09['structured_FE_observations'],
    'unrelaxed_structures':geometries09['registered_models'],
    'geometry_descriptor_rows':geometries09['descriptor_rows'],
    'retrieval_corpus_records':len(corpus09),
    'agent_runs_replayed_from_export':len(agent_runs09),
    'positive_development_cases':critic_dev_metrics09['positive_authored_cases'],
    'positive_development_cases_passed':critic_dev_metrics09['positive_authored_cases_passed'],
    'adversarial_development_cases':critic_dev_metrics09['adversarial_authored_cases'],
    'adversarial_development_cases_passed':critic_dev_metrics09['adversarial_authored_cases_passed'],
    'false_advancements_in_authored_negative_cases':false_advance09,
    'false_rejections_in_authored_positive_cases':false_reject09,
    'research_actions_proposed':len(triage09),
    'research_actions_executed':0,
    'open_research_risks':len(risk_register09),
    'independent_DOI_held_out_evaluation_completed':False,
    'independent_scientific_critic_benchmark_completed':False,
    'production_RAG_validated':False,
    'supervised_selectivity_ML_trained':False,
    'MLIP_validated_for_CO2RR':False,
    'human_visual_review_completed':False,
    'catalyst_selectivity_predicted':False,
    'Bayesian_active_learning_implemented':False,
    'human_approval_modified':False,
}
assert summary09['human_verified_experimental_claims'] == 3
assert summary09['adversarial_development_cases_passed'] == 20
assert not any(summary09[k] for k in (
    'production_RAG_validated','supervised_selectivity_ML_trained',
    'MLIP_validated_for_CO2RR','catalyst_selectivity_predicted',
    'human_approval_modified','independent_DOI_held_out_evaluation_completed'
))
report_md09 = f'''# CuSelect-AI — Initial Benchmarking & Scientific Readiness Report

**Campaign:** CU-001  
**Basis:** Offline, read-only audit of existing Notebook 02–08 exports.

## Executive findings

- **Verified experimental evidence:** {len(claims09)} claims from **one** independent publication ({len(papers09)} seed publications); two structured FE observations and one product-ratio observation.
- **Atomic models:** {len(registry09)} geometrically registered, unrelaxed Cu/Ag slabs with {len(features09)} geometry-only descriptor rows; human visual approval and independent reference calculations remain pending.
- **RAG prototype:** {len(corpus09)} source-linked records. Lexical retrieval and authored smoke checks were demonstrated, but independent retrieval quality, citations on unseen questions, and production readiness have **not** been established.
- **Agent workflow:** {len(agent_runs09)} exported bounded deterministic demonstrations; not autonomous scientific discovery.
- **Scientific Critic:** {4-false_reject09}/4 authored positive cases and {20-false_advance09}/20 authored adversarial cases passed upon independent file-level recomputation. These are **development checks**, not external performance estimates.
- **Active-learning precursor:** {len(triage09)} human-directed research proposals; **zero tasks executed** and no learned acquisition function.

## Evidence, computation and benchmark boundaries

- **No** DFT references or calibrated energy/force models for the eight Cu/Ag geometries.
- **No** experimentally validated mapping from these model structures to CO₂RR Faradaic efficiency or ethylene/ethanol selectivity.
- **No** valid DOI-held-out benchmark can be made from only one approved evidence-bearing publication.
- **No** justified catalyst ranking, independent scientific critic benchmark, or production-ready RAG certification.

## Highest-priority next scientific work

1. Source-review more independent experimental publications.
2. Perform human visual review of all eight geometric slab structures.
3. Validate experimental conditions and comparability by product, electrolyte and operating potential.
4. Acquire independent reference computations after structural review and method definition.
5. Pre-register an independent, DOI-separated RAG/Critic benchmark.

## Integrity

RAG, agent and critic manifest hashes matched the exported files; the human-reviewed register and approved export matched across {len(all_fields09)} source fields per approved claim. This demonstrates data lineage, **not** external truth or model validity.

**Status:** Prototype and evidence audit complete; independent scientific benchmarking remains **pending**.
'''
print('IN-MEMORY RESEARCH REPORT: GENERATED')
print(json.dumps(summary09,indent=2))
display(Markdown(report_md09))
print('No report files written')


IN-MEMORY RESEARCH REPORT: GENERATED
{
  "campaign_id": "CU-001",
  "notebook": "09_Benchmarking_and_Reporting",
  "benchmark_status": "initial_cross_notebook_audit_only",
  "source_publications_seeded": 8,
  "independent_publications_with_verified_claims": 1,
  "human_verified_experimental_claims": 3,
  "structured_FE_claims": 2,
  "unrelaxed_structures": 8,
  "geometry_descriptor_rows": 8,
  "retrieval_corpus_records": 3,
  "agent_runs_replayed_from_export": 3,
  "positive_development_cases": 4,
  "positive_development_cases_passed": 4,
  "adversarial_development_cases": 20,
  "adversarial_development_cases_passed": 20,
  "false_advancements_in_authored_negative_cases": 0,
  "false_rejections_in_authored_positive_cases": 0,
  "research_actions_proposed": 6,
  "research_actions_executed": 0,
  "open_research_risks": 8,
  "independent_DOI_held_out_evaluation_completed": false,
  "independent_scientific_critic_benchmark_completed": false,
  "production_RAG_validated": false,
  "supervis

# CuSelect-AI — Initial Benchmarking & Scientific Readiness Report

**Campaign:** CU-001  
**Basis:** Offline, read-only audit of existing Notebook 02–08 exports.

## Executive findings

- **Verified experimental evidence:** 3 claims from **one** independent publication (8 seed publications); two structured FE observations and one product-ratio observation.
- **Atomic models:** 8 geometrically registered, unrelaxed Cu/Ag slabs with 8 geometry-only descriptor rows; human visual approval and independent reference calculations remain pending.
- **RAG prototype:** 3 source-linked records. Lexical retrieval and authored smoke checks were demonstrated, but independent retrieval quality, citations on unseen questions, and production readiness have **not** been established.
- **Agent workflow:** 3 exported bounded deterministic demonstrations; not autonomous scientific discovery.
- **Scientific Critic:** 4/4 authored positive cases and 20/20 authored adversarial cases passed upon independent file-level recomputation. These are **development checks**, not external performance estimates.
- **Active-learning precursor:** 6 human-directed research proposals; **zero tasks executed** and no learned acquisition function.

## Evidence, computation and benchmark boundaries

- **No** DFT references or calibrated energy/force models for the eight Cu/Ag geometries.
- **No** experimentally validated mapping from these model structures to CO₂RR Faradaic efficiency or ethylene/ethanol selectivity.
- **No** valid DOI-held-out benchmark can be made from only one approved evidence-bearing publication.
- **No** justified catalyst ranking, independent scientific critic benchmark, or production-ready RAG certification.

## Highest-priority next scientific work

1. Source-review more independent experimental publications.
2. Perform human visual review of all eight geometric slab structures.
3. Validate experimental conditions and comparability by product, electrolyte and operating potential.
4. Acquire independent reference computations after structural review and method definition.
5. Pre-register an independent, DOI-separated RAG/Critic benchmark.

## Integrity

RAG, agent and critic manifest hashes matched the exported files; the human-reviewed register and approved export matched across 17 source fields per approved claim. This demonstrates data lineage, **not** external truth or model validity.

**Status:** Prototype and evidence audit complete; independent scientific benchmarking remains **pending**.


No report files written


## 9.12 — Manual, non-overwriting report export (disabled by default)

**First run with `EXPORT_REPORT09 = False`.** Once Sections 9.1–9.11 pass and you have inspected the report, deliberately change it to `True` for **one run**. The output is published only after validating saved files in an isolated staging directory; an existing output folder is never overwritten. Then restore `EXPORT_REPORT09 = False` and save the notebook.

The export is a reporting artifact only. It does not write to source evidence, create new labels, or grant scientific approval.


In [18]:
# PSEUDOCODE
# 1. Keep report export OFF on ordinary Run All executions.
# 2. When manually enabled, refuse to overwrite an existing report folder.
# 3. Write the report, tables, JSON summary and a source-hash inventory into staging.
# 4. Verify every staged file and output digest before publishing the folder.
# 5. Never mutate the scientific records or approval statuses.

EXPORT_REPORT09 = False
REPORT_EXPORT_ROOT09 = ROOT / 'outputs/reports/09_benchmarking'
if not EXPORT_REPORT09:
    print('NOTEBOOK 09 REPORT EXPORT: DISABLED — no files written')
    print('Optional destination:',REPORT_EXPORT_ROOT09)
else:
    assert not REPORT_EXPORT_ROOT09.exists(), 'Existing report export; refusing overwrite.'
    REPORT_EXPORT_ROOT09.parent.mkdir(parents=True,exist_ok=True)
    stage_parent09=Path(tempfile.mkdtemp(prefix='.09_benchmark_staging_',dir=str(REPORT_EXPORT_ROOT09.parent)))
    try:
        stage09=stage_parent09 / REPORT_EXPORT_ROOT09.name
        stage09.mkdir()
        (stage09/'09_scientific_readiness_report.md').write_text(report_md09,encoding='utf-8')
        (stage09/'09_benchmark_summary.json').write_text(json.dumps(summary09,indent=2)+'\n',encoding='utf-8')
        scorecard09.to_csv(stage09/'09_evaluation_scorecard.csv',index=False)
        risk_register09.to_csv(stage09/'09_research_risk_register.csv',index=False)
        triage09.to_csv(stage09/'09_research_triage_snapshot.csv',index=False)
        # Hash the inputs as they exist during the export; this is not evidence of earlier immutability.
        source_hashes09={k:sha_file09(p) for k,p in sorted(paths09.items())}
        (stage09/'09_source_sha256.json').write_text(json.dumps(source_hashes09,indent=2)+'\n',encoding='utf-8')
        output_files09=sorted(p for p in stage09.iterdir() if p.is_file())
        out_hashes09={p.name:sha_file09(p) for p in output_files09}
        result_manifest09={
            'campaign_id':'CU-001','notebook':'09_Benchmarking_and_Reporting',
            'scope':'offline_initial_cross_notebook_audit',
            'scientific_benchmark_completed':False,
            'independent_RAG_benchmark_completed':False,
            'production_ready':False,
            'human_approval_modified':False,
            'report_files_sha256':out_hashes09,
            'input_count':len(paths09),
        }
        (stage09/'09_report_manifest.json').write_text(json.dumps(result_manifest09,indent=2)+'\n',encoding='utf-8')
        assert json.loads((stage09/'09_benchmark_summary.json').read_text(encoding='utf-8')) == summary09
        assert len(pd.read_csv(stage09/'09_evaluation_scorecard.csv')) == len(scorecard09)
        assert len(pd.read_csv(stage09/'09_research_risk_register.csv')) == len(risk_register09)
        assert all(sha_file09(stage09/name)==digest for name,digest in out_hashes09.items())
        assert result_manifest09['scientific_benchmark_completed'] is False
        stage09.rename(REPORT_EXPORT_ROOT09)
    finally:
        shutil.rmtree(stage_parent09,ignore_errors=True)
    print('NOTEBOOK 09 REPORT EXPORT COMPLETE:',REPORT_EXPORT_ROOT09)
    print('Source files and human-review records unchanged')


NOTEBOOK 09 REPORT EXPORT: DISABLED — no files written
Optional destination: /Users/nsolebna/202609_Ironhack/Week 8/Project 3/CuSelect-AI/outputs/reports/09_benchmarking


## 9.13 — Read-only independent inspection of an optional report export

If Section 9.12 remains disabled and there is no previous export, this section skips gracefully. If the output exists, compare saved report manifests to actual bytes and preserve negative readiness flags.


In [14]:
# PSEUDOCODE
# 1. If no report was exported, report a safe skip.
# 2. Otherwise validate every saved output against the report manifest.
# 3. Verify the machine-readable summary still denies scientific validation.
# 4. Check source-hash inventory matches the current source files.
# 5. Do not edit or recreate the report.

if not REPORT_EXPORT_ROOT09.is_dir():
    print('REPORT EXPORT RELOAD: SKIPPED — no manual export present')
else:
    m09 = json.loads((REPORT_EXPORT_ROOT09/'09_report_manifest.json').read_text(encoding='utf-8'))
    assert m09['scientific_benchmark_completed'] is False
    assert m09['production_ready'] is False
    assert m09['human_approval_modified'] is False
    for name,digest in m09['report_files_sha256'].items():
        assert sha_file09(REPORT_EXPORT_ROOT09/name) == digest, name
    saved09=json.loads((REPORT_EXPORT_ROOT09/'09_benchmark_summary.json').read_text(encoding='utf-8'))
    assert saved09 == summary09, 'Existing report describes a different source state; never overwrite silently.'
    snapshot09=json.loads((REPORT_EXPORT_ROOT09/'09_source_sha256.json').read_text(encoding='utf-8'))
    assert set(snapshot09)==set(paths09)
    assert all(snapshot09[k]==sha_file09(path) for k,path in paths09.items()), 'Source files changed since report export.'
    print('REPORT EXPORT INDEPENDENT RELOAD: PASSED')
    print('Verified report files:',len(m09['report_files_sha256']))
print('No files modified')


REPORT EXPORT RELOAD: SKIPPED — no manual export present
No files modified


## 9.14 — Final scientific interpretation and project handoff

Notebook 09 completes the **initial, reproducible research-software prototype roadmap**, not the scientific discovery program. Legitimate next work: more independently reviewed literature, human structure inspection, reference DFT, calibrated MLIP benchmarking, a frozen DOI-separated RAG/Critic evaluation, and controlled model-backed agents. Only thereafter should the project claim validated catalyst selection or autonomous improvement.

**Safe Git checkpoint:** Keep `EXPORT_REPORT09 = False`, save the notebook, then inspect `git status --short` and stage only `notebooks/09_Benchmarking_and_Reporting.ipynb`. Exported source data and reports are separate outputs and may be ignored by Git.


In [17]:
# PSEUDOCODE
# 1. Confirm all cross-notebook checks completed in this kernel.
# 2. Summarize exact progress and the unachieved scientific validations.
# 3. Indicate optional reporting export state without altering any files.

final09={
    'campaign_id':'CU-001',
    'notebooks_00_to_09_initial_roadmap':'initial_prototype_stage',
    'required_source_artifacts_checked':len(paths09),
    'approved_claims_synchronized':len(claims09),
    'registered_geometries':len(registry09),
    'saved_agent_runs_audited':len(agent_runs09),
    'critic_positive_authored_checks_passed':4-false_reject09,
    'critic_negative_authored_checks_passed':20-false_advance09,
    'scientific_benchmark_ready':False,
    'production_RAG_validated':False,
    'MLIP_and_selectivity_validated':False,
    'active_learning_acquisition_implemented':False,
    'source_approval_changed':False,
    'optional_report_export_exists':REPORT_EXPORT_ROOT09.is_dir(),
}
assert final09['approved_claims_synchronized']==3
assert final09['registered_geometries']==8
assert final09['critic_negative_authored_checks_passed']==20
assert final09['scientific_benchmark_ready'] is False
print('NOTEBOOK 09 — INITIAL BENCHMARKING AND REPORTING SUMMARY')
print(json.dumps(final09,indent=2))
print('Roadmap prototype audited; independent scientific validation still required')


NOTEBOOK 09 — INITIAL BENCHMARKING AND REPORTING SUMMARY
{
  "campaign_id": "CU-001",
  "notebooks_00_to_09_initial_roadmap": "initial_prototype_stage",
  "required_source_artifacts_checked": 18,
  "approved_claims_synchronized": 3,
  "registered_geometries": 8,
  "saved_agent_runs_audited": 3,
  "critic_positive_authored_checks_passed": 4,
  "critic_negative_authored_checks_passed": 20,
  "scientific_benchmark_ready": false,
  "production_RAG_validated": false,
  "MLIP_and_selectivity_validated": false,
  "active_learning_acquisition_implemented": false,
  "source_approval_changed": false,
  "optional_report_export_exists": true
}
Roadmap prototype audited; independent scientific validation still required
